# 23. 台灣：資料、模型、缺口與下一步

{doc}`第 22 章 <22_operational_systems>`結束在一個不太像統計課的
結論上：一條 1989 年的公式到今天還在多國作業系統裡跑，模型本身
的進步其實有限；真正費力的是資料的即時品質、制度的權威性、以及
機率該怎麼說出口。從第三步起，瓶頸就不再是統計。

這一章把鏡頭轉回台灣，看看那句話在自己家裡是什麼樣子。台灣是
地球上做統計地震學條件最好的地方之一：板塊聚合速率每年約八公分、
地震多到不缺樣本、觀測網密度世界前段、目錄免費公開（你在
{doc}`第 2 章 <02_download>`就申請過了）。條件好，代表這裡沒有
「資料不夠」這個藉口——缺口如果存在，是別的原因造成的。

本章不推導任何新式子，用到的全是前面留下來的工具：第 11 章的
$M_c$ 與 GR 律、第 12 章的 Omori–Utsu 與 Båth、第 14 章的 ETAS
八參數、第 16 章的 PPE 基準模型、第 17–18 章的一致性檢驗與統計
功效、第 21 章的 PSHA。八節的順序是：盤點資料家底（23.1）、
把散落各章的在地數值收成速查表（23.2）、用三個真實序列看這些
數字怎麼運作（23.3）、對 2024 年 0403 花蓮做一次誠實的事後
檢視（23.4）、列出四個缺口（23.5）、給一份台灣版 CSEP 實作
指南（23.6）、談制度選擇（23.7），最後收束全書（23.8）。

## 23.1 資料家底

談模型之前先談資料，這是第 11 章反覆講的順序。台灣的地震資料
家底可以拆成四份：地震目錄、矩張量目錄、強震紀錄、地表形變與
地球物理連續觀測。

### 兩個目錄數字，兩個不同的基準

你會在不同地方看到兩個差很多的數字，兩個都對，但**基準完全
不同**，混用會鬧笑話：

- 中央氣象署的地震目錄**全期累積超過 67 萬筆**：涵蓋 1900 年
  以來（更早的部分靠歷史文獻與震度反推）、各種規模尺度、以及
  大量 $M_L$ 2.0 以下的微小事件。
- 本書使用的公開目錄是 **1973 年起、$M_L \ge 2.0$** 的版本，
  約 35 萬筆：起始年是儀器目錄的實質起點（TTSN 啟用），規模
  下限是這份公開版本的收錄門檻。

差距不是資料遺失，是**取樣規則不同**。引用時務必連同基準一起
說，否則「台灣有 67 萬筆地震」與「我用了 35 萬筆」放在同一段裡，
讀者只會覺得你搞丟了 32 萬筆。先看這份目錄的全貌：

In [ ]:
from gdms_toolkit.viz import setup_plotly
setup_plotly()

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from gdms_toolkit import load_taiwan_catalog
from gdms_toolkit.viz import ACCENT, PALETTE, QUAKE_COLOR, apply_layout

cat = load_taiwan_catalog()
yearly = cat.set_index("time").resample("YE").size()

# 兩個標竿事件直接由目錄取出（目錄時間為 UTC，0403 主震落在 04-02 UTC）
def biggest(t0, t1):
    win = cat[(cat.time >= t0) & (cat.time < t1)]
    return cat.loc[win.ML.idxmax()]

chichi = biggest("1999-09-20", "1999-09-22")
hualien = biggest("2024-04-02", "2024-04-04")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.58, 0.42],
                    vertical_spacing=0.05)
big = cat[cat.ML >= 5.0]
fig.add_trace(go.Scattergl(x=big.time, y=big.ML, mode="markers", name="ML ≥ 5",
                           marker=dict(size=4, color=ACCENT, opacity=0.5)),
              row=1, col=1)
fig.add_trace(go.Bar(x=yearly.index, y=yearly.values, name="年事件數（全目錄）",
                     marker_color=PALETTE[2]), row=2, col=1)
for ev, label in [(chichi, "集集"), (hualien, "0403 花蓮")]:
    for r in (1, 2):
        fig.add_vline(x=ev.time, line_dash="dot", line_color=QUAKE_COLOR,
                      row=r, col=1)
    fig.add_annotation(x=ev.time, y=7.45, text=f"{label} ML {ev.ML:.1f}",
                       showarrow=False, font=dict(color=QUAKE_COLOR),
                       row=1, col=1)
fig.add_annotation(x="1994-06-01", y=yearly.max() * 0.92, xanchor="right",
                   text="1994 觸發式→連續記錄（儀器）", showarrow=False,
                   font=dict(size=11), row=2, col=1)
fig.update_yaxes(title_text="規模 ML", range=[4.8, 7.7], row=1, col=1)
fig.update_yaxes(title_text="年事件數", row=2, col=1)
apply_layout(fig,
             title=f"台灣長期目錄總覽（1973–2025，ML ≥ 2 共 {len(cat):,} 筆；"
                   f"其中 ML ≥ 5 共 {len(big):,} 筆）",
             height=580, hovermode="x")
fig

這張圖同時寫著兩部歷史，讀圖時要一直分辨哪一部在說話。

**來自地球的**：上圖 1999 與 2024 兩根近乎垂直的柱子，是集集與
0403 花蓮的餘震序列——大地震在幾天內把幾十年份的 $M \ge 5$
事件塞進同一個時間點；下圖對應年份的兩根尖峰是同一件事。

**來自儀器的**：下圖 1994 年前後那道階梯。年事件數翻了好幾倍，
不是台灣突然多地震，而是 CWA 的即時資料由**觸發式改為連續記錄**，
年偵測數從約 4,000 跳到約 20,000。至於 2012 年觀測網升級成熟後
的第二次翻倍（到約 4 萬），主要發生在 $M_L$ 2.0 以下，這份公開
目錄裡看不出來——**你看不見的改善也是必須知道的改善**。

第 11 章的功課在自己的資料上一目瞭然：讀任何目錄統計圖，先問
哪些特徵來自地球、哪些來自儀器。台灣目錄該切的五刀，第 11.1
節已經列過（1973、1987.6、1991.3、1994、2012），這裡只提醒一件
事：**現代的台灣 ETAS 研究幾乎一律從 1994 年起算**。

### 三條規模轉換式，與它們對 $b$ 值的影響

台灣目錄的「規模」欄位換過三次定義，跨年代比較之前必須先均一化。
第 11.1 節擁有這三條式子，這裡只複述結果與用途：

$$\begin{aligned}
M_w &= 0.87\,M_L + 0.23 &&\text{(AutoBATS)}\\
M_D &= 0.187 + 0.862\,M_L &&\text{(Wang et al. 1989)}\\
M_L &= -0.24 + 1.07\,M_w \pm 0.31 &&\text{(Chang et al. 2016)}
\end{aligned}$$

第一條量化了「$M_L$ 在大規模端偏高」這件事：$M_L\,6.0$ 對應
$M_w \approx 5.45$、$M_L\,7.0$ 對應 $M_w \approx 6.32$。把 CWA
的 $M_L$ 目錄直接餵進以 $M_w$ 校準的模型，觸發率與規模分布都會
系統性偏掉。第二條的用途則不是換算單一事件，而是**換算 $b$ 值**：
GR 律在兩個尺度下分別寫成 $\log_{10}N = a' - b' M_D$ 與
$\log_{10}N = a - b M_L$，把 $M_D = 0.187 + 0.862 M_L$ 代進去，
立刻得到 $b = 0.862\,b'$（23.3 節的池上個案會用到）——**不換算
就比較 $b$ 值，比較的是儀器不是地殼**。

三條式子救不回的是**飽和**。$M_D$ 在 6.0 以上飽和、$M_L$ 在 6.5
以上飽和，1978 與 1986 兩個真實 $M_w$ 7 以上的事件在原目錄被嚴重
低估，最大誤差可達一個規模單位。線性迴歸修不了飽和；Chang et al.
(2016) 的作法是改用**優先序**（Harvard $M_w$ > USGS > 轉換後的
BATS $M_w$ > 新轉換 $M_w$），並補進 188 個 1900–1935 年的遺漏
事件、補平 1985–1991 年 $M \ge 5$ 的資料空隙——這解釋了為什麼
台灣需要一份寬頻矩張量目錄。

### AutoBATS CMT：規模飽和的答案

BATS 寬頻網 1992 年啟動、1996 年由中研院地球所建置完成，目前
台灣區域有 35 個永久寬頻站。**AutoBATS** 是它的自動矩張量反演
流程：CWA 警報觸發後自動擷取波形，用 FK 法建立的格林函數資料庫
掃描信噪比、地殼模型、頻帶、方位角與距離涵蓋、等向性百分比、
質心深度等條件，自動挑出最佳解。1996–2023 年共產出
**8,171 個解**，其中 7,189 個品質等級優於 C3。

品質怎麼驗證？拿 1996–2016 年間三千多個與 Global CMT 的共同解
比對：機制解的 **Kagan 角平均差 $22.0° \pm 16.6°$**、$M_w$ 平均差
$-0.08 \pm 0.10$。Kagan 角是兩個雙偶極機制之間的最小旋轉角，22°
給了一把可用的尺——**機制解自帶二十度上下的不確定度**。

### TSMIP 與 TGNS：波形與形變

**TSMIP**（台灣強地動觀測計畫）目前有 737 個自由場強震站與 45
座結構物陣列，解析度已由 16 位元全面升級到 24 位元（地動振幅
解析能力提升約 200 倍）；三十年累積逾 28,000 個地震、22 萬筆紀錄，
其中 1999 年集集的數十筆車籠埔近斷層紀錄讓全球近斷層資料一舉
增加數倍。這份資料是
{doc}`第 21 章 <21_psha>`裡 GMPE 的原料，也是 23.3 節大埔預報
能一路算到「場址震度機率」的原因。**TGNS** 則是氣象署的地球
物理觀測網：163 個 GNSS 站、6 個地下水位站、12 個地磁站、20 個
大地電場站——第一部{doc}`第 3 章 <03_groundwater>`到
{doc}`第 6 章 <06_gnss>`用的就是這些資料。CWBSN 本身則有超過 170
個測站，含 62 個平均深度 300 m 的井下站與東部海域的海底地震儀站。
把四份資料放在一起看，台灣的家底是**齊的**：目錄、機制、波形、
形變都有，且都公開——後面 23.5 節要列的四個缺口，沒有一個是
「資料不夠」。

## 23.2 在地基準值總表

第 11、12、14 章的每一條定律，台灣都有自己的數字。這一節把它們
收成速查表，先講清楚三件事：這些數值**由各章擁有**，本節只彙總
與指路，不重新推導也不重新擬合；每一列都有**但書欄**，而速查表
最危險的用法就是把數字抄走、把但書留下；表的深層讀法是第二部的
方法論總復習——每一條但書都是前面某一章的核心教訓。

### 表一：$b$ 值

| 量 | 台灣值 | 但書 | 章節 |
|---|---|---|---|
| 全台平均 | 1.0–1.1 | 隨時段、規模尺度、深度變動 | 11.2 |
| 1973–1987 / 1994–2011 | 0.83 / 0.99 | 差異來自尺度換代，非構造 | 11.1 |
| 空間型態 | 陸上高、外海低 | 等值線走向與構造一致，需長時段才穩 | 11.3 |
| 集集餘震 | 0.84 | $a = 6.4$，$2 < m < 5$ 擬合 | 12.5 |
| 池上前震 / 餘震 | 0.52–0.62 / 0.84–0.87 | 兩個估計法差 0.1，都低於背景 0.95 | 11.4 |

「1973–1987 的 0.83 與 1994 後的 0.99」是全表最該記住的一列：
差 0.16 看起來像構造變化，其實是 $M_D \to M_L$ 的尺度換代——
**這個量級的落差，換一次規模尺度就足以製造出來**。

### 表二：$M_c$ 與 Omori $p$

| 量 | 台灣值 | 但書 | 章節 |
|---|---|---|---|
| $M_c$ 空間場 | 陸上 1.5–2.0、外海 2.5–3.2 | 用完整 CWBSN 目錄；公開目錄看不到 1.5 | 11.3 |
| $M_c$ 分年代 | 2.61（1973–87）／2.40（1994–2011） | EMR 法，每 500 事件一窗 | 11.1 |
| 主震後 $M_c(t)$ | 初期可達 4.0，隨 $\log t$ 回落 | 在這條線以下統計的是儀器 | 11.3 |
| $p$（集集） | 1.05（$c = 6\times10^{-3}$ 天） | 目錄拼接了 TSMIP 與 CMT | 12.3 |
| $p$（池上，6／12／30 天窗） | 1.39／1.30／0.92 | 同一份資料，只換時間窗 | 12.3 |

另外一條要單獨列，因為它顛覆的是教科書講法：Tsai et al. (2012)
用台灣目錄檢驗多碎形應力活化模型的預測，得到

$$p(M_m) = (0.11 \pm 0.01)\,M_m + (0.38 \pm 0.02),$$

東部 $p \approx 0.11 M_m + 0.36$、西部 $p \approx 0.07 M_m + 0.50$，
三種除叢法結果幾乎一致。也就是說 **$p \approx 1$ 不是普世常數，
它至少在統計上隨主震規模增加**（完整討論見 12.3 節）。但這些
係數在不同目錄之間**並未**與任何構造特徵相關——它是穩健的
經驗關係，不是有物理解釋的定律。

### 表三：Båth 與 ETAS

| 量 | 台灣值 | 但書 | 章節 |
|---|---|---|---|
| Båth $\Delta_1$（706 序列） | $1.20 \pm 0.73$ | 與 Båth 原值一致，但 $\sigma$ 極大 | 12.5 |
| Båth（餘震數 > 50） | $0.74 \pm 0.52$ | $\Delta_1$ 隨餘震數下降 | 12.5 |
| Båth（主震 $M \ge 6$） | $1.26 \pm 0.43$ | 與上一列方向相反，分組決定答案 | 12.5 |
| Båth（集集，GR 外推） | $\Delta M^* = 0.03$ | 實測 0.95；小餘震異常多 | 12.5 |
| $\Delta M$ 對 $M_m$ 迴歸 | $-1.01 + 0.31\,M_m$ | 原作者聲明不可用於預測 | 12.5 |

注意 Båth 這一組有多亂：1.20、0.74、1.26、0.03、外加一條斜率
0.31 的迴歸線。**這種混亂本身就是教學重點**——「常數 1.2」高度
依賴序列的餘震數、主震規模門檻與 $M_c$ 的選擇，而 $\sigma = 0.73$
意味著 68% 區間橫跨 1.5 個規模單位（能量差約 180 倍），這種區間
對疏散決策沒有幫助。

ETAS 的部分，台灣有**兩份互相獨立的八參數估計**：CWA 112 年
（2023）委辦報告子計畫三（訓練窗 1994–2021）與 2025 年大埔快報
（訓練窗 1994–2024/12），兩者都取 $M_c = M_L\,3.0$、都用 Cheng
et al. (2015) 的淺層地殼分區。完整對照表在 14.9 節，這裡只收結論：

| 參數群 | 兩份估計 | 但書 | 章節 |
|---|---|---|---|
| $p$、$\alpha$、$\mu$ | $p$ 1.035／1.062；$\alpha$ 1.04／1.17 | 被資料釘死，交叉驗證成功 | 14.9 |
| $\gamma$、$q$、$D$ | $\gamma$ 0.35／0.68；$q$ 2.43／1.59 | 空間核鬆脫；$D$ 與 $D^2$ 記號不同不可直接比 | 14.3 |
| $\mu$ 的意義 | 0.5424／0.5098 | 可能是率密度，也可能是鬆弛係數，先確認 | 14.9 |
| 分支比 $n$ | 兩組代入都得 $n > 1$ | 不代表超臨界，是慣例落差 | 13.4 |

這張表比任何文字都能說明 14.3 節的主張：ETAS 的八個參數不是
平等的，時間核那兩三個被資料釘得很死，空間核那三個鬆得很。
兩份獨立估計的離散度是唯一的不確定度來源——原報告沒有標準誤。

### $M_c$ 是一張地圖：自己算一次

表二第一列說 $M_c$ 是空間場，值得自己算一遍。用 0.2° × 0.2° 網格、
1994 年後、深度 ≤ 30 km 的事件，逐格套第 11.3 節的最大曲率法
（MAXC + 0.2）；關鍵是**別逐格去掃全目錄**，正確作法是一次
`np.histogramdd` 把三維直方圖建好，再沿規模軸取 `argmax`：

In [ ]:
DEG, DM = 0.2, 0.1
MIN_N = 50                       # 每格至少要有這麼多事件才估 Mc

lon_e, lat_e = np.arange(119.4, 123.21, DEG), np.arange(21.4, 26.01, DEG)
mag_e = np.arange(1.95, 7.05, DM)
shallow = cat[(cat.time >= "1994") & (cat.depth <= 30)]

# 三維直方圖：一次掃完，不對 35 萬筆做逐格比對
H, _ = np.histogramdd((shallow.longitude.to_numpy(), shallow.latitude.to_numpy(),
                       shallow.ML.to_numpy()), bins=(lon_e, lat_e, mag_e))
n_cell = H.sum(axis=2)
mag_c = np.round(mag_e[:-1] + DM / 2, 2)
mc_grid = np.where(n_cell >= MIN_N, mag_c[np.argmax(H, axis=2)] + 0.2, np.nan)

lon_c, lat_c = lon_e[:-1] + DEG / 2, lat_e[:-1] + DEG / 2
LO, LA = np.meshgrid(lon_c, lat_c, indexing="ij")
on_land = (LO > 120.0) & (LO < 121.9) & (LA > 22.0) & (LA < 25.2)  # 粗略本島方框
mc_land, mc_sea = np.nanmean(mc_grid[on_land]), np.nanmean(mc_grid[~on_land])

fig = go.Figure(go.Heatmap(x=lon_c, y=lat_c, z=mc_grid.T, colorscale="Blues",
                           zmin=2.2, zmax=3.5, colorbar=dict(title="Mc"),
                           hovertemplate="%{x:.1f}°E %{y:.1f}°N<br>"
                                         "Mc=%{z:.1f}<extra></extra>"))
for ev, name in [(chichi, "1999 集集"), (hualien, "2024 0403 花蓮")]:
    fig.add_trace(go.Scatter(x=[ev.longitude], y=[ev.latitude], mode="markers",
                             name=name,
                             marker=dict(symbol="x", size=12, color=QUAKE_COLOR,
                                         line=dict(width=2))))
apply_layout(fig,
             title=f"Mc 空間場（0.2° 網格，1994 年後，深度 ≤ 30 km，"
                   f"每格 ≥ {MIN_N} 筆；本島平均 {mc_land:.2f}、"
                   f"其餘 {mc_sea:.2f}）",
             xaxis_title="經度", yaxis_title="緯度",
             yaxis_scaleanchor="x", hovermode="closest", height=560)
fig

圖上的對比很清楚：本島與近岸一片淺色，外海——尤其東部與東北
外海——明顯偏深。這正是 Chan & Wu (2013) 用完整 CWBSN 目錄得到
的圖像：測站涵蓋差的地方，偵測不到的小地震就多，$M_c$ 就高。

但這張圖有一個**必須講出來的限制**，而且它剛好是第 11 章那句
「估計方法只能看到資料讓它看到的東西」的活教材：本圖的 $M_c$
最低只到 2.2，而文獻報告的陸上值可以低到 1.5。差別不在方法，
在資料——**本書使用的公開目錄本身就在 $M_L\,2.0$ 被左截斷**，
MAXC 的峰值最低只能落在 2.0，加上 0.2 的保守修正就是 2.2。
陸上真實的偵測能力比這張圖顯示的更好，我們只是看不到。

這個限制不影響圖的主要用途，對比的**方向與量級**仍然可信：本島
與外海平均差約 0.3 個規模單位，文獻用完整目錄得到的落差則達 1.7
個單位——依 GR 律那對應約 50 倍的事件數差異。對全島套單一 $M_c$
等於讓外海的不完整資料污染整體統計，這正是兩份 ETAS 估計明知
陸上可用到 2.0 仍保守取 $M_c = 3.0$ 的理由。

## 23.3 三個序列個案

表格會讓人以為統計地震學是一組常數。三個真實序列可以把這個錯覺
打破：同樣的定律、同一個國家、二十六年之內，行為差得很遠。

### 1999 集集：Båth 定律的破產點

集集 $M_w\,7.65$（目錄 $M_L\,7.3$）是台灣現代地震學的分水嶺。
Lee et al. (2013) 的分析在方法上有一個值得抄走的細節：**目錄
拼接**。CWBSN 目錄當主幹，用 TSMIP 強震紀錄補主震後第一小時被
尾波淹沒的早期餘震，再用 CMT 補 $m \ge 5$ 的大餘震——這正是第
11.3 節說的 STAI（主震後短期不完整性）的實務解法：不是把早期
資料丟掉，而是換一個不會飽和的資料源補進來。

補齊之後，主震後 1000 天內區內記錄到 **42,952 個 $m \ge 2.0$
餘震**，其中 9 個 $m \ge 6.0$。三條定律的擬合結果是
$\log_{10} N(\ge m) = 6.4 - 0.84\,m$（$2 < m < 5$），即
$b = 0.84$、$a = 6.4$；Omori 部分 $p = 1.05$、
$c_0 = 6 \times 10^{-3}$ 天，而集集前六年的背景事件間隔為
$\tau_b = 0.1753$ 天（背景率約 5.7 次/天）。

真正的教學價值在 Båth。實測最大餘震 $m_w\,6.70$，
$\Delta m = 7.65 - 6.70 = 0.95$，看起來很正常。但改用 GR 外推的
**推論最大餘震** $m^* = a/b = 6.4/0.84 = 7.62$，得到
$\Delta M^* = 0.03$——遠低於典型的 0.8–1.5。這個數字的意思是
**集集的小餘震數量異常多**：若套加州平均值 $\Delta M^* = 1.11$，
主震規模得是 8.73 才配得上這麼多小餘震。作者推測與破裂含有
無震／慢地震成分有關。

為什麼 $m^* = a/b$ 比「觀測到的最大餘震」更值得看？因為後者是
單一極值樣本、方差極大（第 12.5 節用順序統計量算過
$D_1 \sim \mathrm{Exp}(\beta)$），前者用的是整個 GR 分布的資訊。

### 2022 池上：前震、時間窗，與兩個都低的 $b$ 值

2022 年 9 月的台東序列有一個集集沒有的東西：**顯著的前震**。
9/17 關山 $M_L\,6.6$（$M_w\,6.55$），隔天 9/18 池上主震
$M_L\,6.8$（$M_w\,6.95$）。整個序列的地震波能量幾乎全由這兩個
事件釋放。前震區比餘震區小，餘震由前震區向外擴展。

$b$ 值的結果是國際上最被關注的前兆候選之一在台灣的一個支持案例：

| 分組 | 最小二乘 | 最大概似 | 對照背景 |
|---|---|---|---|
| 前震 | 0.62 | 0.52 | 0.95 |
| 餘震 | 0.87 | 0.84 | 0.95 |
| 全序列 | 0.71 | 0.65 | 0.95 |

三件事要一起讀。**其一**，前震 $b$ 值明顯低於餘震，差 0.25（LSQ）
到 0.32（MLE），與 Wetzler et al. (2023) 在全球八個區域得到的
「前震 $b$ 值低 0.1–0.2」方向一致。**其二**，背景值 0.95 是
換算出來的——文獻早年用 $M_D$ 給出本區 $b' \approx 1.1$，經
$M_D = 0.187 + 0.862 M_L$ 換算得 $0.862 \times 1.1 = 0.95$；
沒有這一步，前震與餘震的 $b$ 值看起來只是「低於 1.1」，換算後
才看得出**兩者都低於背景**。**其三**，最小二乘與最大概似差 0.10
（前震段差更多）——估計法本身就是不確定度來源。

但最重要的一句話是：**這是回溯分析，不是即時判定**。9/17 當天
沒有人知道那是前震；要在 9/17 就宣告「這個 $b = 0.52$ 代表大
地震要來了」，需要的是一個事前註冊、對所有序列一律適用的規則，
而不是事後在已知答案的情況下算出來的一個數字。池上的另一個
貢獻是 $p$ 值：同一份目錄、同一支程式，全期 30 天得 $p = 0.92$，
取前 12 天得 1.30，只取前 6 天得 **1.39**（完整式子
$n(t) = 27.7/(t + 0.02)^{0.92}$）。第 12.3 節解釋過機制：真實
序列不是單一 Omori 核，窗一拉長涵蓋到大餘震，那個大餘震自己的
次級序列就把尾巴撐高，單一核唯一能做的事就是把 $p$ 調小。
**報告 $p$ 值時不附上擬合時間窗，等於沒報告**。

### 2025 大埔：一條完整的作業 pipeline

2025/01/20 大埔 $M_L\,6.4$（深度 15.8 km，大埔震度 6 弱）之後，
台灣第一次公開發表了端到端的作業型餘震預報實測（Hsieh et al.
2025）。到 2/07 共觸發 162 個 $M_L \ge 3.0$ 餘震，其中 8 個
$\ge 5.0$。整條流程值得逐步拆解，因為它把第 11 到 21 章串成了
一條線：

1. **目錄**：GDMS 目錄 1973/01–2024/12/29 為底，但只取 1994 年
   以後訓練（第 11.1 節的那一刀）；2024/12/30 起改用 CWA 即時
   速報，因為作業系統等不到重定位目錄。
2. **$M_c$ 與空間範圍**：$M_c = M_L\,3.0$、深度 ≤ 35 km 的淺層
   地殼分區、81 × 201 格點，訓練樣本 64,239 個 $M_L \ge 3$
   事件（第 11.3 節）。
3. **ETAS 參數**：MLE + AIC，6 次迭代收斂，得
   $\mu = 0.5424$、$A = 0.9166$、$c = 0.0012$ 天、
   $\alpha = 1.0408$、$p = 1.0350$、$D^2 = 0.0007$ deg²、
   $q = 2.4253$、$\gamma = 0.3511$（第 14.1、14.9 節）。
4. **算力**：128 核、6 次 MLE 迭代約 1 小時完成訓練；沿用預訓練
   參數後，每個即時預報時窗只需 **8 核、7 分鐘**。
5. **1,000 份合成目錄**：用估好的 $\lambda^*$ 逐時窗模擬（第 10
   章的反函數法、第 13 章的分支模擬），把預報的不確定性直接用
   蒙地卡羅表示，而不是靠解析近似。
6. **GMM 與場址震度機率**：對每一份合成目錄的每一個模擬事件，
   用地動預估式（Lin et al. 2011，含 $V_{S30}$ 與孕震深度）算
   各站 PGA、轉震度，再統計各場址的超越機率（PoE），每小時
   更新一次（第 21 章）。

這條 pipeline 的每一段都不新——新的是**它們接起來了，而且跑得
動**。第 4 步那組數字尤其值得記住：作業化的瓶頸在預訓練而非即時
運算。更難得的是這份報告**把預報機率與實際觀測並列發表**（以
震後 17:00 UTC 起算）：

| 時窗 | $P(M_L\ge5)$ | $P(M_L\ge6)$ | $P(M_L\ge7)$ | 實際 $\ge5$ 次數 |
|---|---|---|---|---|
| 1 天 | 30.5% | 3.4% | 0.1% | 1 |
| 3 天 | 42.1% | 6.2% | 0.2% | 1 |
| 7 天 | 58.8% | 8.1% | 0.3% | 7 |
| 10 天 | 67.8% | 11.2% | 0.3% | 8 |

把它畫出來，機率與次數兩條軸的對比更清楚：

In [ ]:
win_label = ["1 天", "3 天", "7 天", "10 天"]
p_ge5 = [30.5, 42.1, 58.8, 67.8]      # P(ML ≥ 5)，%
p_ge6 = [3.4, 6.2, 8.1, 11.2]         # P(ML ≥ 6)，%
p_ge7 = [0.1, 0.2, 0.3, 0.3]          # P(ML ≥ 7)，%
obs_ge5 = [1, 1, 7, 8]                # 實際發生的 ML ≥ 5 次數
obs_ge6 = 0

fig = make_subplots(specs=[[{"secondary_y": True}]])
for name, vals, color in [("P(ML ≥ 5)", p_ge5, PALETTE[0]),
                          ("P(ML ≥ 6)", p_ge6, PALETTE[1]),
                          ("P(ML ≥ 7)", p_ge7, PALETTE[3])]:
    fig.add_trace(go.Bar(x=win_label, y=vals, name=name, marker_color=color),
                  secondary_y=False)
fig.add_trace(go.Scatter(x=win_label, y=obs_ge5, mode="lines+markers",
                         name="實際 ML ≥ 5 次數",
                         line=dict(color=QUAKE_COLOR, width=2.5, dash="dot"),
                         marker=dict(size=11, symbol="diamond")),
              secondary_y=True)
fig.update_yaxes(title_text="至少發生一次的機率（%）", range=[0, 80],
                 secondary_y=False)
fig.update_yaxes(title_text="實際發生次數", range=[0, 10], showgrid=False,
                 secondary_y=True)
apply_layout(fig,
             title=f"2025 大埔序列：ETAS 預報機率 vs 實際觀測"
                   f"（10 天內實際 ML ≥ 5 共 {obs_ge5[-1]} 次、"
                   f"ML ≥ 6 共 {obs_ge6} 次）",
             xaxis_title="自震後 17:00 UTC 起算的時窗",
             barmode="group", hovermode="x", height=470)
fig

表與圖都必須小心讀，因為它們很容易被當成「評分表」。**機率與
次數是兩種不同的量**：長條是「時窗內至少發生一次的機率」，紅色
菱形是「實際發生了幾次」，兩者不能直接對齊比高低。可以合法說出
口的觀察有兩個。第一，次數的成長（1→8）比機率的成長（30.5%→
67.8%）陡，暗示模型在這個序列上可能偏保守，但**單一序列不足以
下這個結論**。第二，$P(M_L \ge 6)$ 從 3.4% 爬到 11.2% 而實際一次
都沒發生；**這不算預報失敗**，11.2% 的事件本來就有 88.8% 的機會
不發生。要判斷機率預報準不準，必須用一整批預報做一致性檢定
（N-test、S-test、負二項或二元似然，第 17 章）與比較檢驗（資訊
增益，第 18 章）——這是全書最容易被違反的一條原則。

序列本身還有兩個誠實面。**其一**，主震前源區沒有明顯高活動率，
即時目錄未顯示任何前驅訊號——**大埔沒有前兆**。**其二**，
序列呈四階段結構（主震 → 1/25 $M_L\,5.7$ → 1/30 $M_L\,5.6$ →
之後），每隔約 5–6 天出現一次群集；1/24 到 1/25 之間有一段「在
Omori 框架下不尋常的平靜」，但作者謹慎地指出**即時目錄的完整度
與定位精度不足以判定那是不是真平靜**，23.5 節會把這個落差列成
第三個缺口。場址端則有另一個教訓：嘉義站（近）的震度
超越機率曲線清楚呈現四階段結構，機率峰值常對應到後續實測到的
較高震度，台南站（遠）就模糊得多——**距離調節了餘震對場址
震度的影響**。

三個序列合起來說的是同一件事。集集告訴我們 Båth 的「常數」可以
崩到 0.03；池上告訴我們 $p$ 值可以在同一份資料上從 0.92 變到
1.39；大埔告訴我們一整條 pipeline 可以在 7 分鐘內跑完一輪。
**定律給的是分布，不是數值**——速查表列的是分布的中心，個案展示
的是分布的寬度，把中心當成答案用遲早會翻車。

## 23.4 0403 花蓮的事後檢視

第一部{doc}`第 7 章 <07_case_hualien2024>`對 2024 年 0403 花蓮
$M_L\,7.2$ 做過四類觀測資料的對照，結論是乾淨的訊號只有波形與
餘震。現在有了第二部的工具，可以問一個更精確的問題：**如果 0403
當天台灣有一套完整的三尺度預報系統，它會說什麼？**

這一節分長期、短期、中期三段回答，每一段的答案都不一樣，而且
沒有一段是「差一點就能預測」。

### 長期：PPE 看見了，但那不算預報

先做一件只用 0403 **之前**資料就能做的事：用 1973 年到 2024 年
3 月底的目錄，畫一張第 16.6 節的 PPE 空間項 $h_0$——「與過去
地震的鄰近性」——再把 0403 震央疊上去。震央座標直接由目錄取，
不手打：

In [ ]:
D_KM, S_BG, MC_PPE = 15.0, 1e-4, 5.0        # 沿用 16.6 節的 h0 參數
pre = cat[(cat.time < hualien.time) & (cat.ML >= MC_PPE)]

lons, lats = np.arange(119.0, 123.51, 0.1), np.arange(21.0, 26.01, 0.1)
LON, LAT = np.meshgrid(lons, lats)
dens = np.zeros_like(LON)
for lo, la, mi in pre[["longitude", "latitude", "ML"]].to_numpy():
    r2 = ((LON - lo) * 111 * np.cos(np.radians(LAT))) ** 2 \
         + ((LAT - la) * 111) ** 2
    dens += (mi - MC_PPE + 0.1) * (1 / (np.pi * (D_KM ** 2 + r2)) + S_BG)

# 0403 震央落在全島率密度的哪個百分位？
j = int(np.argmin(np.abs(lats - hualien.latitude)))
i = int(np.argmin(np.abs(lons - hualien.longitude)))
pct = (dens < dens[j, i]).mean() * 100

fig = go.Figure(go.Heatmap(x=lons, y=lats, z=np.log10(dens), colorscale="Blues",
                           colorbar=dict(title="log₁₀ 相對率"),
                           hovertemplate="%{x:.1f}°E %{y:.1f}°N<extra></extra>"))
fig.add_trace(go.Scatter(x=[hualien.longitude], y=[hualien.latitude],
                         mode="markers", name=f"0403 花蓮 ML {hualien.ML:.1f}",
                         marker=dict(symbol="x", size=15, color=QUAKE_COLOR,
                                     line=dict(width=3))))
apply_layout(fig,
             title=f"只用 0403 之前的目錄（{len(pre):,} 個 ML ≥ "
                   f"{MC_PPE:.0f} 事件）：震央落在率密度第 {pct:.0f} 百分位",
             xaxis_title="經度", yaxis_title="緯度",
             yaxis_scaleanchor="x", hovermode="closest", height=560)
fig

答案是：**看見了，但這算不上預報**。0403 震央落在全島率密度的
高百分位區——花蓮外海本來就是全台地震活動度最高的地帶之一，
任何一個長期平滑地震度模型都會把最高的率放在那裡。這正是第
16.6 節講 PPE 的三個角色時說過的：基準模型回答「哪裡」很在行，
對「何時」完全沉默——它的率密度只隨新事件加入而緩慢更新，在
2024 年 3 月 31 日與 4 月 1 日給的地圖幾乎一樣。0403 落在高背景
區，是基準模型的成功，也是它的天花板——**一個永遠正確但永遠
不改變的答案，資訊量是零**。這也是為什麼第 18 章要用資訊增益
而不是命中率來評分：要贏 PPE，你必須額外說出「何時」。

### 短期：ETAS 型 OAF 事前不會示警，事後立刻接手

0403 主震前，即時目錄上**沒有顯著的前震序列**——這與 2022 池上
形成鮮明對比（池上有 $M_L\,6.6$ 的關山前震在前一天）。一套 ETAS
型的作業化餘震預報（OAF）系統在 0403 之前不會發出任何警示，這
不是系統失靈，而是**設計如此**：ETAS 的條件強度
$\lambda^*(t,x,y,m) = \mu(x,y) + \sum_{i:t_i<t} \kappa(m_i)
g(t-t_i) f(x-x_i,y-y_i;m_i)$，沒有觸發事件就只剩背景項。但主震
之後完全是另一回事：同一套系統可以立刻接手，而且我們知道它會
表現得不錯——23.3 節那條大埔 pipeline 的每一個零件都是通用的，
預訓練參數、$M_c$、GMM、7 分鐘一輪的算力都已經到位。**0403
之後台灣缺的不是技術能力**。

### 中期：這一格是空白的

誠實的回答是：**沒有東西可以檢視**。台灣目前沒有作業化的中期
（月到十年）模型，所以「0403 前一年，中期模型會說什麼」這個
問題連問都問不了——這是 23.5 節第一個缺口的具體樣貌。

### 事後總能找到跡象，這正是問題所在

這次檢視最大的價值，是它把{doc}`第 8 章 <08_explore_ideas>`的
警告變成具體的東西。事後回頭看，你**總是**能在圖上找到「跡象」：
挑一個夠寬的空間窗、一段夠長的時間、一個夠自由的統計量，再在
已知答案的情況下調整這三個選擇，直到訊號浮出來。第 18 章給了它
一個名字與一套解藥：名字是**自由度**——事後分析裡分析者可以動
的旋鈕，遠多於他報告出來的那幾個；解藥是**零自由度的前瞻檢驗**
——模型參數、預報規格、目標資料來源全部在觀測之前明確定義並
公開註冊，之後不准改。CSEP 整個社群存在的理由就是這個。

所以 0403 教我們的不是「差一點就能預測」，而是**各時間尺度的
工具各自能做什麼、不能做什麼**：長期模型知道哪裡、不知道何時；
短期模型知道大震之後、不知道大震之前；中期那一格，台灣還沒有
工具。

## 23.5 四個缺口

把台灣現有的能力攤在時間軸上，缺口的位置就一目瞭然：

In [ ]:
YR = 365.25
tools = [
    ("地震預警 EEW", np.log10(3 / 86400 / YR), np.log10(60 / 86400 / YR),
     "#1baf7a", "作業中（世界前段班）｜本書範圍外"),
    ("短期預報 ETAS/OAF", np.log10(1 / YR), np.log10(90 / YR),
     "#2a78d6", "技術就緒（第 13–14 章）"),
    ("中期預報", np.log10(0.25), np.log10(20),
     "#e34948", "缺口（第 15–16 章的工具尚未在地化）"),
    ("長期危害 TEM PSHA", np.log10(10), np.log10(500),
     "#4a3aa7", "兩代國家級模型（第 20–21 章）"),
]
fig = go.Figure()
for name, lo, hi, color, status in tools:
    fig.add_trace(go.Bar(y=[name], x=[hi - lo], base=[lo], orientation="h",
                         marker_color=color, opacity=0.85, name=status,
                         text=status, textposition="inside",
                         insidetextanchor="middle",
                         hovertemplate=f"{name}<br>{status}<extra></extra>"))
fig.add_annotation(x=np.log10(1.5), y=3.62, showarrow=False,
                   text="檢驗文化（第 17–18 章）貫穿全部尺度",
                   font=dict(size=11, color="#555"))
fig.update_xaxes(title_text="時間尺度（年，log₁₀）", tickvals=[-6, -4, -2, 0, 2],
                 ticktext=["秒–分", "小時", "天", "年", "百年"])
apply_layout(fig, title="台灣地震風險資訊工具的時間尺度地圖與第二部章節對應",
             showlegend=False, hovermode="closest", height=420)
fig

### 缺口一：中期模型

圖上最刺眼的是中間那一段。短期有 ETAS（天到週，兩份在地參數、
一次實戰驗證），長期有 TEM PSHA（數十年到數百年，兩代國家級
模型，第 21.4 節），中間**月到十年**這一段沒有在地化的作業模型。
這一段不是可有可無的：它正好對應災後重建的中期規劃（基督城
重建用的就是 50 年時變危害模型）與既有建物補強的優先順序
（紐西蘭 Kaikōura 之後的強制補強政策直接建立在中期模型上）。
缺了這一格，短期預報算出來的機率沒有辦法接上長期危害，只能各
說各話。第 15–16 章介紹的 Ψ 現象與 EEPAS 正是填這一格的工具族；
EEPAS 的台灣在地化工作正在進行中，在成果正式發表之前，這一格
在圖上只能標成紅色。

### 缺口二：預報檢驗文化尚未建立

這是四個缺口裡最不像缺口、也最關鍵的一個，因為它不是缺一個模型，
是缺一套**制度**。台灣目前的預報相關研究絕大多數是**個案回溯
分析**：某個序列發生了，事後套模型、報參數、討論擬合品質。這類
工作有價值，但無法回答「這個模型比別的模型好嗎」，因為分析者是在
看過答案之後才決定要看什麼。2025 大埔快報之所以難得，是因為那份
預報**在事件發生過程中逐時窗發出**，不是事後補算的。缺的是 CSEP 式、事先註冊的前瞻檢驗：模型與
參數在期初鎖定並公開，目標事件的定義（規模門檻、深度範圍、
時空箱、要不要除叢）在期初講清楚，然後等時間過去，用第 17 章的
一致性檢驗與第 18 章的比較檢驗評分——23.6 節就在講怎麼做。

### 缺口三：即時目錄與重定位目錄的落差

作業型預報用的是**即時目錄**，研究用的是**重定位目錄**，兩者的
統計性質不同——這個落差在教科書上很少被提及，但對實務極重要。
具體證據在 23.3 節：大埔快報的作者指出 1/24 到 1/25 那段「不尋常
的平靜」無法據即時目錄定論。這不是謙虛，是可量化的問題——CWA
112 年報告的 RMT 系統與 CWA 速報／重定位結果比對 429 筆事件，
規模差平均 $-0.39$、發震時間差平均 0.91 秒、深度差平均 4.6 km、
水平震央差平均 4.1 km，編號地震的偵測率約 69%。這幾個數字的
後果是連鎖的：規模系統性偏差會直接搬移 $M_c$ 與 $b$ 值（第 11
章）；偵測率不足會讓早期餘震缺漏，而第 14.6 節證明過，缺漏
事件會讓 $\alpha$ 被低估、$c$ 被高估、$p$ 被拉低。**用重定位
目錄訓練、用即時目錄預報**，這個錯配在每一個作業系統裡都存在，
台灣也不例外。

### 缺口四：前兆研究的誠實現況

最後一個缺口，數字由業務單位自己提供。CWA 的地震前兆觀測業務
分中長期（震前約半年，看地震活動時空變化與地殼形變）與短期
（震前數小時到 10 天，看電離層 TEC、地下水位、地磁、大地電場）。
業務回顧的原話是：真正成功的案例非常有限，尤其短期前兆
「**可視為成功發現前兆的比例似乎都在 2 成以下**」。

這句話不是「前兆研究沒用」，也不是「地震完全不可預測」，正確
讀法是第一部整整八章的結論：訊號太小、樣本太少、事後選擇太會
騙人。2016 美濃地震前，中南部山區的地震個數與能量釋放都低於長期
背景的第一四分位、台南歸仁—仁德的 GNSS 基線由每年縮短約 1 cm
轉為「鎖住」——這些事後看起來很像前兆的訊號確實存在，但沒有一個
通過過前瞻檢驗。所以這個缺口不是放棄的理由，而是**資源配置的
理由**：把力氣優先投給已被證明有預報技巧的統計叢集模型，同時把
前兆研究放進同一套檢驗框架（第 18 章的 Molchan 圖與面積技能分數
就是為警報式模型設計的）。

值得停下來看一眼：這四個缺口，**沒有一個能靠發明新模型來補**。
中期模型的工具早就存在（第 15–16 章），缺的是在地化與驗證；
檢驗文化缺的是制度與紀律；目錄落差缺的是工程與資料治理；前兆
缺的是誠實的評分標準。這正好呼應第 22 章的結論——從第三步起，
瓶頸就不再是統計。

## 23.6 若要在台灣做 CSEP

前一節說檢驗文化是最關鍵的缺口。這一節把它變成可執行的清單：
**如果要在台灣建一個 CSEP 型的前瞻檢驗實驗，有哪些坑要先繞開**。
全部材料來自第 17–18 章，這裡做的是在地化。

### 先問一個逆風的問題：在地模型一定比較好嗎

開始之前先接受一個檢驗結果。Bayona et al. (2023) 把全球模型
**GEAR1**（平滑地震度 × 板塊間應變率的乘法組合，以 1977–2013
全球 CMT 的 $M_w\,5.767+$ 淺震校準）投影到加州、紐西蘭、義大利
三個 CSEP 測試區，與 19 個區域時間獨立模型做前瞻對決
（2014–2022，$M\,4.95+$），結果 GEAR1 在紐西蘭排第 1、加州第 2、
義大利第 3。這三個地方是全世界儀器覆蓋最密、研究最透徹的地震
區，區域模型卻沒能穩定打敗一個用全球資料訓練出來的模型。所以
「**我們的模型是為台灣量身打造的，所以一定比較好**」是一個
**需要被檢驗的假設，不是公理**。這也直接給了台灣一個現成的
基準：把 GEAR1 投影到台灣測試區，任何新的台灣區域模型都應該
先證明自己贏得過它，再談別的。

### 坑一：測試區小、目標事件少，功效會低到沒有意義

這是台灣最該擔心的問題。Khawaja et al. (2023) 做過一個令人不安的
實驗：在慣用的 0.1° × 0.1° 全球網格上，一個「地球上每個地方機率
都一樣」的**均勻模型竟然通過了 S-test**——不是模型好，是檢驗
沒有功效：648 萬個格子配 651 顆地震，平均一萬格才一顆，空間資訊
被解析度稀釋掉了。作者估計要在那種設定下得到有力的檢驗，需要
32,000 顆以上的地震，約當 300 年的記錄。

台灣的情況更緊。用 23.1 節的轉換式把國際慣用的目標門檻
$M_w\,4.95$ 換到台灣的 $M_L$ 尺度：由 $M_w = 0.87 M_L + 0.23$
反解得 $M_L \approx 5.4$。在本書使用的目錄裡，取 CWA ETAS 研究
慣用的測試區（119.8–122.5°E、21.7–25.6°N、深度 ≤ 35 km），
2017–2024 這八年共有約 100 顆 $M_L \ge 5.4$ 的事件。一百顆聽
起來還行，但**它們不是一百個獨立樣本**：其中 47 顆發生在 2024
年，而光是 0403 序列的頭三個月就貢獻 39 顆。這就是義大利的
翻版——義大利八年只有 11 顆目標地震，兩份研究的模型排名之所以
不同，主要就因為 2012 Emilia 序列（單一序列貢獻 11 顆
$M\,4.95+$）落在或不落在測試期內。**單一序列可以翻轉整份排名**。

實務結論有三條：(a) 任何「台灣模型通過檢驗」的宣稱都必須附上
該檢驗的**功效估計**（第 18.5 節），否則「沒被拒絕」什麼也沒說；
(b) 測試期要夠長，或明確報告單一序列的影響；(c) 網格設計要當成
實驗設計的一部分認真對待。

### 坑二：等寬網格會殺掉檢驗力，改用 Quadtree

上面那個問題有一半是網格造成的，而網格是我們自己選的。Khawaja
等人的一維合成實驗最能說明：固定 20 個箱時，**等期望率分箱**
只要 15 顆事件就達到功效 0.9，**等寬分箱**要 38 顆——同一批
資料，只是換了分箱方式，檢驗力就差了一倍以上。

**Quadtree** 是這個想法的二維版本：整個區域當根節點，每個格子
要嘛不分、要嘛切成四個子格，以「每格最多容納的資料點數
$N_{\max}$」與「最大縮放層級 $L_{\max}$」控制，讓地震多的地方
細、少的地方粗；在多解析度網格上，S-test 只要 8 顆地震就能達到
最大功效。網格之間的轉換有理論保證：聚合是把小格的率相加、
反聚合是把大格的率均勻攤到小格，而 Poisson 相加仍是 Poisson
（率為 $\Lambda = \sum \lambda_i$），所以**真模型聚合後仍會通過
檢驗，假模型則會在新網格上被抓出來**。對台灣這種「陸上密、外海
疏」的分布，多解析度網格幾乎是必要的——均勻網格會把大量幾乎
沒有地震的海域格子塞進似然裡，稀釋掉真正有資訊的那幾格。

### 坑三：台灣序列的叢集極強，Poisson 假設會低估變異

23.3 節的三個序列已經說明了問題：集集 1000 天內 42,952 個
$m \ge 2$ 餘震、0403 序列三個月貢獻全年目標事件的大半。這種分布
下，「每個時空箱的事件數服從 Poisson」會嚴重低估變異——
Poisson 的變異數等於期望值，而叢集資料的變異數遠大於期望值。
後果是 N-test 的信賴區間太窄，模型動不動就被拒絕，而拒絕的
原因是叢集不是模型錯。第 17 章給了兩條出路，兩條都適合台灣：

- **負二項似然**：換成有額外離散度參數的負二項分布，讓變異數
  可以大於期望值。代價是那個參數本身要從資料估——在目標事件
  少的台灣，這件事不便宜。
- **二元似然（binary likelihood）**：只問「這個格子有沒有發生
  至少一顆」，一次把叢集造成的計數膨脹整個拿掉，代價是丟棄
  計數資訊。Bayona 等人跨區比較時就同時報告 Poisson 的「每顆
  地震分數」與二元的「每個 active cell 分數」。

台灣兩者都該報：**只在 Poisson 似然下贏的勝利，很可能是叢集
造成的假象**。

### 坑四：對齊。規模尺度、深度、除叢，一個都不能少

這一條看起來最無聊，但它是「apples-to-apples」的全部內容，也
剛好把本章第一節接回來：**23.1 的三條轉換式在這裡變成檢驗設計
的必要條件**。

- **規模尺度與門檻**：GEAR1 用全球 CMT 的 $M_w$ 校準、以
  $M_w\,5.95+$ 發布，台灣的目標目錄是 $M_L$、門檻是 4.95+。
  兩層換算都要做：尺度換算用 $M_w = 0.87 M_L + 0.23$（義大利
  的前例是把率除以 1.602），門檻下放要用 $b$ 值外推，而台灣的
  $b$ 不是 1（表一）——這個選擇必須事先宣告，不能等看到結果
  再回頭調。也要記得 $M_L$ 在 6.5 以上飽和。
- **深度**：GEAR1 涵蓋到 70 km，台灣的短期模型通常只取深度
  ≤ 35 km 的淺層地殼分區。Bayona 等人的作法值得抄：**實證檢查
  深部事件的貢獻可忽略**（紐西蘭 40–70 km 只佔 6%、義大利 30 km
  以下掛零）之後才宣稱可比。台灣的隱沒帶深震比例明顯高於這兩地，
  這個檢查不能省，結論很可能是「不可忽略，必須分開測試」。
- **除叢**：時間獨立模型（PPE、GEAR1、PSHA 背景）應該對除叢
  目錄測試，叢集模型（ETAS）應該對完整目錄測試。第 12.6–12.7
  節講過除叢本身帶著選擇效應，用哪一套演算法、哪一組連結參數
  （台灣慣用 5 km／3 天）必須寫進實驗規格。

### 坑五：預先註冊與零自由度

最後一條是整套制度的地基。CSEP 誕生的直接原因是地震預測研究
長期缺乏**可重現性**與**可複製性**，而同儕審查不足以保證這兩件
事；解方是把所有模型參數、預報規格、目標資料來源都在觀測之前
明確定義，達成**零自由度的獨立檢驗**。落到台灣的實作，最低限度
是四件事：模型與參數在期初鎖定並存檔（附版本號）；目標事件的
完整定義（規模尺度與門檻、深度範圍、時空箱、除叢與否、用哪一版
目錄）寫成公開文件；基準模型至少一個（PPE 或投影到台灣的
GEAR1）；分析程式與資料打包成可重現套件。

最後補一句立場。CSEP 社群自己的說法是：**不因為模型沒通過某一項
檢驗就正式否決它**，而是把分位數分數當作診斷工具。這個態度對
台灣特別重要——在一個目標事件這麼少的測試區，把單次檢驗結果
當成生死判決，只會逼出更多過度自信的宣稱。**檢驗是為了學到
東西，不是為了頒獎**。

## 23.7 制度選擇

技術清單列完了。剩下的問題全部不是技術問題。

### 三條路線，三種對「公開」的答案

第 22.2 節比較過三個作業化系統的引擎與節奏，這裡只看制度面
那一欄——那才是台灣真正要做的選擇題：

| | 對誰發布 | 制度重點 |
|---|---|---|
| 義大利 INGV | 僅民防體系，不對大眾 | 候選模型必須提交過 CSEP 檢驗才能進官方集成 |
| 紐西蘭 GNS | 完全公開 | 預報與使用者共同設計，直接接上補強政策 |
| 美國 USGS | 完全公開 | 大震後自動發布，貝氏更新，不確定性隨資料收窄 |

三條路線各自解決了不同的問題。**義大利**把「誰有資格進入官方
系統」制度化了——「開發者自己相信」不算數，必須通過第三方前瞻
檢驗。**紐西蘭**把「機率算給誰看」制度化了——預報表每一格都
同時給平均數、範圍、機率與判讀語彙，而且基督城重建的 50 年
時變危害模型與 Kaikōura 之後的強制補強政策都建立在它上面。
**美國**把「什麼時候發」制度化了——大震後自動觸發，不需要有人
臨場決定要不要說話。值得注意的是這三件事**互相獨立**：台灣可以
公開發布但不自動觸發、或自動觸發但只給民防、或要求 CSEP 檢驗
但暫不公開。這不是三選一的套餐。

### 台灣既有的基礎設施

台灣不是從零開始，有兩塊基礎已經很紮實。**第一塊是預警**。
CWA 自 1993 年開始發展地震預警，是國際先驅，里程碑很具體：
1999 年集集地震，速報系統在大規模停電下仍於**震後 102 秒**算出
位置與規模，當時是全球地震速報的里程碑；2018 年花蓮地震，
**震後 17 秒發布警報、20 秒**經手機與電視觸達民眾，5 分鐘正式
地震報告，7 分鐘鄉鎮市區尺度的細緻化震度，10 分鐘內成立災害
應變中心。2013 年起全國 3,500 所中小學安裝地震資訊接收軟體，
台大的 P-Alert 微機電觀測網到 2023 年 8 月已建置 777 站。

但務必記住{doc}`第 9 章 <09_forecasting_intro>`的區辨：**預警是
「地震已經發生、波還在路上」，預報是「地震還沒發生」**。前者
是通訊與訊號處理問題，後者是統計問題。台灣在預警上世界領先，
完全不代表在預報上領先——這兩件事共用觀測網，但幾乎不共用
任何方法論。把兩者混為一談，是台灣公眾溝通上最常見的誤解。

**第二塊是發布管道**。CWA 的地震事件頁、細胞廣播與電視台蓋台
插播機制，都是經過大量實戰的發布基礎設施——三條路線裡最花錢的
那一段（怎麼把訊息送到每個人手上）台灣已經蓋好了，缺的是
**要送什麼內容、由誰決定送不送**。

### 權威性由誰授予

這是第 22 章留下的問題，也是本章唯一無法用資料回答的問題。機率
預報要能發揮作用，發布者必須具備權威性——而權威性不是技術指標，
是社會授予的。2009 年義大利 L'Aquila 地震之後的審判讓全世界的
地震學家記住了一課，但這一課常常被記錯：不是「說了會被告」，而是
科學家與官員被批評的恰恰是**沒有把「機率確實升高了」講得夠清楚、
夠強**——沉默的代價與說錯話的代價同樣真實。

台灣要回答的具體問題有三個，每一個都不是統計學家能單獨決定的：
誰是法定的發布者？發布的內容是機率、情境敘事，還是行動建議？
當機率升高但仍然很小時，發布的門檻在哪裡？第 22.3 節整理過的
溝通要點——用「未來一週**之內**」而非「下一週」、把機率換成
頻率框架、報數量範圍比報機率範圍抗扭曲、機率永遠配上文字判讀
——都是有實驗支持的技術，但技術解決不了「誰有資格說話」。那是
制度問題，而制度問題的答案不會從資料裡長出來。

## 23.8 全書結語

這本教材從地下水位的固體潮講起，繞了一大圈，結束在制度設計與
一張畫著紅色缺口的時間尺度地圖上。回頭看，兩部其實在講同一件
事的兩面。

**第一部教的是懷疑的紀律**。地下水位、地磁、GNSS、波形——每一類
都有人宣稱過它能預測地震，而每一次仔細追下去，訊號都在雜訊、
儀器、氣象與事後選擇裡溶解掉。第 7 章對 0403 花蓮做的四類資料
對照，結論是乾淨的訊號只有波形與餘震；第 8 章把方法論講白了：
異常這種東西，只要你願意找，永遠找得到。

**第二部教的是把懷疑制度化**。同樣的懷疑精神只停在「我不相信」
就只是犬儒。第 9 到 23 章走的是另一條路：先承認有一種可預報性
是真的（叢集——大地震之後短期內地震率確實升高數十倍，反覆被
檢驗、被多國作業系統每天使用），再建立一整套機制讓真假宣稱
可以被分辨。基準模型讓「有技巧」有了對照組；前瞻檢驗讓事後
選擇無所遁形；資訊增益讓「比較好」變成一個有單位的數字；統計
功效讓「通過檢驗」這件事本身也要被檢驗。

這套機制的代價是它讓人謙虛。**我們能做到的**：對一個大地震之後
的時空範圍，給出經得起檢驗的餘震機率與場址震度機率；把這些機率
接上地動預估式與工程決策；用資訊增益公平比較兩個模型的優劣；
在世界上好幾個地方每天運轉這套流程——1989 年之前，這些話沒有
一句說得出口。**我們做不到的**：告訴你下一個大地震的時間與地點；
判定眼前的 $b$ 值下降是不是前震訊號；用單一序列的結果證明一個
模型好或不好；可靠地在中期尺度上發言（至少在台灣還不行）；以及
最根本的——把絕對機率提高到有人願意據以疏散的水準。

台灣的位置很特別：世界級的資料、世界級的預警、正在成形的短期
預報、一段還沒有人填上的中期空白、以及一套還沒開始運轉的檢驗
文化。這四件事裡，只有第一件是老天給的。

那段空白，也許就留給正在讀這一頁的你。工具在前面十四章裡，目錄
就在 `data/cache/` 裡，缺口在 23.5 節列得清清楚楚，而 23.6 節
那份清單就是入場的規則。動手之前，只需要記得全書說了兩遍的那句
話——**一個數字算得出來，不等於它站得住腳**。讓它站住腳的方法，
你現在已經會了。